In [1]:
verbose = False

In [2]:
from load_dataset import get_data
import torch
from torch.utils.data import TensorDataset
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("GPU" if torch.cuda.is_available() else "CPU")
X_raw, y = get_data(verbose=True, device=device)

CPU
Loading dataset


                               | 0/480 [          ]

Dataset loaded, X : torch.Size([480, 32, 3200])


In [3]:
if verbose:
    from load_dataset import labelList
    from ml_util.eeg_util import plotEEG
    plotEEG(X_raw[0], title="index 0")

    print(labelList[y[0]])

In [4]:
from preprocess import preprocess_dataset

# Applies preprocessing
X = preprocess_dataset(X_raw, verbose=True)
print(X.shape)

from ml_util.data_module import DataModule 

# Stores in a TensorDataset
dataset = DataModule(X, y)

Checkpoint preprocessed found
torch.Size([480, 32, 5, 101])


In [5]:
if verbose:
    import torch
    import matplotlib.pyplot as plt
    from load_dataset import labelList
    from ml_util.eeg_util import plotTimeFreqEEG
    for label in range(4):
        sampleId = torch.where(y==label)[0][0]
        plt.figure()
        fig, axes = plotTimeFreqEEG(X[sampleId])
        fig.suptitle(f"Sample {sampleId}, with label {labelList[y[sampleId]]}")
    plt.show()

In [12]:
hyperparams = {}
hyperparams['lr'] = 4e-3
hyperparams['weight_decay'] = 0
hyperparams['batch_size'] = 16

from bigmodel import model

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=hyperparams['lr'], weight_decay=hyperparams['weight_decay'])

from ml_util.trainer import Trainer
trainer = Trainer(model, optimizer, loss_fn, hyperparams, device)

def babysitter(self, epoch_results):
    self.hyperparameters['lr'] = 0.9 * self.hyperparameters['lr']



trainer.set_babysitting_callback(babysitter)

In [13]:
trainer.train(dataset, 100)

Epoch    0# Loss/train: 44.639442,  Acc/train: 0.312500   |   Loss/val: 158.572525,  Acc/val: 0.277778
Epoch    1# Loss/train: 64.541214,  Acc/train: 0.187500   |   Loss/val: 130.227753,  Acc/val: 0.250000
Epoch    2# Loss/train: 101.544411,  Acc/train: 0.187500   |   Loss/val: 150.490509,  Acc/val: 0.250000
Epoch    3# Loss/train: 141.333466,  Acc/train: 0.125000   |   Loss/val: 161.619156,  Acc/val: 0.250000
Epoch    4# Loss/train: 132.823181,  Acc/train: 0.250000   |   Loss/val: 162.057297,  Acc/val: 0.250000
Epoch    5# Loss/train: 117.441635,  Acc/train: 0.500000   |   Loss/val: 163.477219,  Acc/val: 0.152778
Epoch    6# Loss/train: 91.263474,  Acc/train: 0.437500   |   Loss/val: 164.722733,  Acc/val: 0.152778
Epoch    7# Loss/train: 85.244560,  Acc/train: 0.437500   |   Loss/val: 156.198929,  Acc/val: 0.152778
Epoch    8# Loss/train: 74.208115,  Acc/train: 0.375000   |   Loss/val: 144.414551,  Acc/val: 0.236111
Epoch    9# Loss/train: 93.387306,  Acc/train: 0.062500   |   Loss/va